# Challenge 4: Answer workflow (Greeter → Search → Critique → Refine)

An agent that answers questions but **verifies and refines the answer before returning it**, built as an ADK graph `Workflow`:

```
START → capture_question → greeter → route_intent ─┬─ "greeting" → send_greeting
                                                    └─ "question" → search → critique → refine
```

| Node | Type | Job |
|---|---|---|
| `capture_question` | function | Saves the user's question in session state |
| `greeter` | agent | Classifies the message: a greeting/small talk, or a real question |
| `route_intent` | function | Routes on the greeter's verdict |
| `send_greeting` | function | Replies to greetings without running the answer team |
| `search_agent` | agent + Google Search | Researches and writes the first draft |
| `critique_agent` | agent | Reviews the draft against the question and lists concrete improvements |
| `refiner_agent` | agent | Rewrites the draft applying every improvement → final answer |

Each agent in a graph receives only the previous node's output, so the agents also save their results to session state (`output_key`) and later agents read them through `{placeholders}` in their instructions: the critic sees the question **and** the draft, the refiner sees the draft **and** the critique.

**To run:** Runtime → Run all. The project is detected from the environment.

In [ ]:
%pip install -q "google-adk==2.7.0"

## Configuration

In [ ]:
import logging
import os
import subprocess

MODEL = "gemini-2.5-flash"
GCP_LOCATION = "global"                 # Gemini on Agent Platform: global endpoint, more capacity than one region


def detect_project_id() -> str:
    """Finds the active Google Cloud project from the environment or gcloud."""
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if not project:
        try:
            import google.auth

            _, project = google.auth.default()
        except Exception:
            project = None
    if not project:
        result = subprocess.run(
            ["gcloud", "config", "get-value", "project"], capture_output=True, text=True
        )
        project = result.stdout.strip()
    if not project:
        raise RuntimeError("No Google Cloud project found. Set GOOGLE_CLOUD_PROJECT.")
    return project


PROJECT_ID = detect_project_id()
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S", force=True
)
for noisy_logger in ("httpx", "google_adk", "google_genai", "google.auth"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)
logger = logging.getLogger("answer_workflow")

print(f"Project: {PROJECT_ID}   Model: {MODEL} ({GCP_LOCATION})")

## Agents

In [ ]:
from typing import Literal

from google.adk import Agent
from google.adk.tools import google_search
from google.genai import types
from pydantic import BaseModel, Field


class Intake(BaseModel):
    """The greeter's structured verdict on the user's message."""

    intent: Literal["greeting", "question"] = Field(
        description="'greeting' for hellos, thanks or 'what can you do'; 'question' for anything to answer."
    )
    reply: str = Field(description="A short friendly reply when intent is 'greeting'; otherwise empty.")


greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL,
    description="Welcomes the user and decides whether the message needs the answer team.",
    instruction=(
        "You are the friendly front desk of a question-answering service. Classify the user's "
        "message. If it is only a greeting, thanks, or a question about what you can do, set "
        "intent='greeting' and write a one- or two-sentence welcome that invites a question "
        "(the service researches answers, then fact-checks and polishes them). Otherwise set "
        "intent='question' and leave reply empty."
    ),
    output_schema=Intake,
    output_key="intake",
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)

search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description="Researches the question with Google Search and writes a first draft.",
    instruction=(
        "Research the user's question with Google Search, then write a complete, practical "
        "first-draft answer.\n\nQuestion: {question}\n\n"
        "Cover the main steps or facts, include specifics (quantities, order of steps, "
        "tools), and list the sources you used at the end."
    ),
    tools=[google_search],
    output_key="draft_answer",
)

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description="Reviews the draft answer and lists concrete improvements.",
    instruction=(
        "You are a demanding editor and fact-checker. Review the draft answer to the question.\n\n"
        "Question: {question}\n\nDraft answer:\n{draft_answer}\n\n"
        "Check accuracy, completeness (missing steps or considerations), safety, organization, "
        "and clarity for a non-expert. Output a numbered list of specific, actionable "
        "improvements, most important first. Do not rewrite the answer yourself. If the draft "
        "is already excellent, say so and suggest only minor polish."
    ),
    output_key="critique",
)

refiner_agent = Agent(
    name="refiner_agent",
    model=MODEL,
    description="Rewrites the draft applying the critique; produces the final answer.",
    instruction=(
        "Rewrite the draft answer so that it applies every improvement in the critique.\n\n"
        "Question: {question}\n\nDraft answer:\n{draft_answer}\n\nCritique:\n{critique}\n\n"
        "Output only the final answer for the user: well organized (short headings or "
        "numbered steps), accurate, and easy to follow. Keep the sources list."
    ),
    output_key="final_answer",
)

## Workflow graph

Function nodes are plain Python. Returning an `Event` lets a node pass `output` to the next node, write `state`, choose a `route`, or send a `message` to the user.

In [ ]:
from google.adk import Context, Event, Workflow
from google.adk.runners import InMemoryRunner


def capture_question(node_input: types.Content) -> Event:
    """START node: store the user's message in state for the agents' instructions."""
    question = "".join(part.text or "" for part in node_input.parts or []).strip()
    return Event(output=question, state={"question": question})


def route_intent(ctx: Context, node_input: dict) -> Event:
    """Route on the greeter's verdict; questions go on to the answer team."""
    intent = node_input.get("intent", "question")
    logger.info("route_intent -> %s", intent)
    if intent == "greeting":
        return Event(route="greeting", output=node_input.get("reply") or "Hello! Ask me a question.")
    return Event(route="question", output=ctx.state.get("question", ""))


def send_greeting(node_input: str) -> Event:
    """Reply to greetings without running the answer team."""
    return Event(message=node_input)


answer_workflow = Workflow(
    name="AnswerWorkflow",
    description="Greets, then answers questions with a Search -> Critique -> Refine team.",
    edges=[
        ("START", capture_question, greeter_agent, route_intent),
        (route_intent, {"greeting": send_greeting, "question": search_agent}),
        (search_agent, critique_agent),
        (critique_agent, refiner_agent),
    ],
)

# For ADK compatibility (adk web / adk run), the root must be named root_agent.
root_agent = answer_workflow

runner = InMemoryRunner(node=root_agent, app_name="answer_workflow")


async def ask(question: str) -> None:
    """Runs one question through the workflow (fresh session) and prints each node's output."""
    session = await runner.session_service.create_session(app_name="answer_workflow", user_id="student")
    print(f"YOU: {question}\n")
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(user_id="student", session_id=session.id, new_message=message):
        text = "".join(part.text or "" for part in (event.content.parts if event.content else None) or [])
        if text.strip():
            print(f"===== {event.author} =====\n{text.strip()}\n")

## Demo

### A greeting is answered by the greeter; the answer team doesn't run

In [ ]:
await ask("Hi there! What can you do?")

### A question goes through Search → Critique → Refine

The output shows each agent in turn: the greeter's verdict, the draft, the critique, and the refined final answer.

In [ ]:
await ask("How do I clean out a closet?")

### Another question, to show the workflow is general

In [ ]:
await ask("How do I organize a small closet that has no shelves?")